In [1]:
# use first lat, last lat, I should really have flights use the last lat, lon in the ADS-B sequence where it thinks there was a flight. 
# Also first lat, last lat has to be on American FAA airspace. 

In [5]:
from datetime import date

from data_engineering.flights.flight_type import get_flights
from flights.evaluation.american_evals import continental_usa_to_continental_usa
import polars as pl

target_date = date(2026,3,15)
df_flights = get_flights(target_date)
df_flights = df_flights.filter(pl.col("ladd") | pl.col("pia"))

all_flight_counts_by_icao = df_flights.group_by("icao").agg(pl.len().alias("all_flight_count"))
continental_flight_counts_by_icao = (
    continental_usa_to_continental_usa(df_flights)
    .group_by("icao")
    .agg(pl.len().alias("continental_flight_count"))
)
continental_only_icaos = (
    all_flight_counts_by_icao
    .join(continental_flight_counts_by_icao, on="icao", how="left")
    .filter(pl.col("all_flight_count") == pl.col("continental_flight_count"))
    .select("icao")
)

df_flights = df_flights.join(continental_only_icaos, on="icao", how="inner")

In [12]:
TELEPORT_MIN_DISTANCE_KM = 50.0
EARTH_RADIUS_KM = 6371.0088

prev_lat = pl.col("prev_last_lat").radians()
prev_lon = pl.col("prev_last_lon").radians()
curr_lat = pl.col("first_lat").radians()
curr_lon = pl.col("first_lon").radians()
haversine_a = (
    ((curr_lat - prev_lat) / 2).sin().pow(2)
    + prev_lat.cos() * curr_lat.cos() * ((curr_lon - prev_lon) / 2).sin().pow(2)
)

teleport_flights = (
    df_flights
    .sort(["icao", "takeoff_time", "landing_time"])
    .with_columns(
        pl.col("landing_time").shift(1).over("icao").alias("prev_landing_time"),
        pl.col("landing_airport_ident").shift(1).over("icao").alias("prev_landing_airport_ident"),
        pl.col("last_lat").shift(1).over("icao").alias("prev_last_lat"),
        pl.col("last_lon").shift(1).over("icao").alias("prev_last_lon"),
    )
    .filter(
        pl.all_horizontal(
            pl.col("prev_last_lat").is_not_null(),
            pl.col("prev_last_lon").is_not_null(),
            pl.col("first_lat").is_not_null(),
            pl.col("first_lon").is_not_null(),
        )
    )
    .filter(
        pl.col("prev_landing_airport_ident").is_null()
        | pl.col("takeoff_airport_ident").is_null()
        | (pl.col("prev_landing_airport_ident") != pl.col("takeoff_airport_ident"))
    )
    .with_columns(
        (2 * EARTH_RADIUS_KM * haversine_a.sqrt().clip(0, 1).arcsin()).alias("endpoint_jump_km"),
        ((pl.col("takeoff_time") - pl.col("prev_landing_time")).dt.total_seconds() / 3600).alias("time_gap_hours"),
    )
    .with_columns(
        (pl.col("endpoint_jump_km") / pl.col("time_gap_hours")).alias("required_ground_speed_kmh")
    )
    .filter(pl.col("endpoint_jump_km") >= TELEPORT_MIN_DISTANCE_KM)
    .select(
        "icao",
        "prev_landing_time",
        "takeoff_time",
        "time_gap_hours",
        "endpoint_jump_km",
        "required_ground_speed_kmh",
        "prev_landing_airport_ident",
        "takeoff_airport_ident",
        "prev_last_lat",
        "prev_last_lon",
        "first_lat",
        "first_lon",
        "pia",
        "ladd",
        "aircraft_type",
        "owner",
    )
    .sort("endpoint_jump_km", descending=True)
)

teleport_icaos = (
    teleport_flights
    .group_by("icao")
    .agg(
        pl.len().alias("teleport_count"),
        pl.max("endpoint_jump_km").round(1).alias("max_endpoint_jump_km"),
        pl.min("time_gap_hours").round(2).alias("min_time_gap_hours"),
        pl.col("takeoff_time").min().alias("first_teleport_takeoff_time"),
    )
    .sort("max_endpoint_jump_km", descending=True)
)

teleport_icaos

icao,teleport_count,max_endpoint_jump_km,min_time_gap_hours,first_teleport_takeoff_time
str,u32,f64,f64,datetime[ms]


In [11]:
df_flights.filter(pl.col("icao") == "ac0da0")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""ac0da0""","""""","""""",2026-03-15 05:49:58.460,"""KIDA""",2026-03-15 07:00:54.860,"""KSLC""",2026-03-15 06:19:38.690,44.13652,-111.714158,25000,24450,2026-03-15 07:00:54.860,40.772964,-111.962097,4075,4225,false,true,false,false,"""PC12""","""INHS LEASING COMPANY LLC""","""PILATUS PC-12""","""A1"""
"""ac0da0""","""""","""""",2026-03-15 08:19:42.025,"""KSLC""",2026-03-15 09:18:58.740,"""KWYS""",2026-03-15 08:24:43.880,40.782041,-111.962107,4025,4200,2026-03-15 09:18:58.740,43.780472,-112.266732,15825,15625,false,true,false,false,"""PC12""","""INHS LEASING COMPANY LLC""","""PILATUS PC-12""","""A1"""
"""ac0da0""","""""","""""",2026-03-15 17:00:16.780,"""KWYS""",2026-03-15 18:40:50.980,"""KAPA""",2026-03-15 17:30:00.010,43.898481,-109.684884,27000,26700,2026-03-15 18:43:30.140,39.576501,-104.849701,null,null,false,true,false,false,"""PC12""","""INHS LEASING COMPANY LLC""","""PILATUS PC-12""","""A1"""
"""ac0da0""","""""","""""",2026-03-15 20:40:11.660,"""KAPA""",2026-03-16 05:31:35.390,"""KUAO""",2026-03-15 20:47:17.800,39.566143,-104.850708,5675,5850,2026-03-16 05:31:35.390,45.255386,-122.768555,-50,200,false,true,false,false,"""PC12""","""INHS LEASING COMPANY LLC""","""PILATUS PC-12""","""A1"""


In [ ]:
df_flights.filter(pl.col("icao") == "ada9bd")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""ada9bd""","""""","""""",2026-03-15 11:17:04.820,"""KINW""",2026-03-15 12:24:33.470,"""KINW""",2026-03-15 11:39:18.770,35.719254,-109.93,18050,18475,2026-03-15 12:26:34.110,33.425903,-112.020845,null,null,false,true,false,false,"""BE9L""","""GUARDIAN FLIGHT LLC""","""BEECH 90 King Air""","""A1"""
"""ada9bd""","""""","""""",2026-03-15 17:46:13.895,"""KINW""",2026-03-15 18:53:15.620,"""KDVT""",2026-03-15 18:11:14.910,35.724289,-110.027618,20075,20650,2026-03-15 18:53:15.620,33.686783,-112.083164,null,null,false,true,false,false,"""BE9L""","""GUARDIAN FLIGHT LLC""","""BEECH 90 King Air""","""A1"""


In [9]:
from data_engineering.adsb.read_adsb import read_adsb
from evaluation.flights.visual import visualize_flight


visualize_flight(df_flights.filter(pl.col("icao") == "a5824e"), read_adsb(target_date, icaos=["a5824e"]))